# Reproducing Qü et al. (2025) from dandiset 001340

Paper: Qü, Tai, Hall, Tu, Eckstein, Mishchanchuk, Lin, Chase, MacAskill, Collins, Gershman and
Wilbrecht (2025), "Nucleus accumbens dopamine release reflects Bayesian inference during
instrumental learning", PLoS Comput Biol 21(7) e1013226, doi:10.1371/journal.pcbi.1013226.

Dataset: DANDI:001340, version 0.250221.0527 (doi:10.48324/dandi.001340/0.250221.0527). The paper
names the dandiset (Data Availability) but no version. This is the only published version; the
draft has the same 69 assets and size and was last modified before it was published, so the most
recent published version is pinned.

Claims: `claims/001340/10.1371-journal.pcbi.1013226.yaml`, written from the version of record
(Europe PMC full text with Methods and the supplementary legends; no supplementary figures).

Model code: the paper's cognitive models (github.com/Wilbrecht-Lab/cogmodels at commit 589555c)
are re-implemented in `qu2025.py` with numba, for speed. The section "Model code check" below
fetches that code at the pinned commit and shows that both give the same likelihood and RPEs on
the dataset's trials. The code's logistic regression (LR) is used as it is.

All 39 claims are tested; none is blocked.

| Claim | Tested |
|---|---|
| data-composition, perf-after-pretraining, fig1f-performance-increase, trials-to-switch, trials-to-commit, figs1-switch-measures-decrease, fig1e-stay-after-omission, figs6a-decaying-history-weights | tested (behaviour) |
| fig2c-bayesian-beat-rl4p, fig2c-complex-rl-beat-rl4p, fig2c-rlcf, fig2c-no-bayes-vs-complex-rl, rl4p-learning-rates, brlfw-alpha-zero, figs7-no-change-across-sessions | tested (model fits) |
| fig2b-model-identifiability, figs2-parameter-recovery, fig2f-adaptation-after-switch, fig3b-sse, fig3a-both-port-contexts, fig3c-aic-around-switch | tested (model simulation) |
| trial-cleaning-counts, reward-switch-reward-count, fig4bc-reward-omission, fig4bc-switch-effect, fig4bc-no-laterality, fig4de-port-duration | tested (dopamine) |
| fig4f-bayesian-rpe-fit, figs3-other-metrics, figs8-brlwrp-no-difference, fig4g-history-decrease, fig4h-rpe-only-llk, fig4h-nonnegative-rpe-aaa, fig4-consistent-after-reward | tested (dopamine and model RPEs) |
| fig4i-lmer-switch, fig4j-lmer-stay, fig4i-model-runchosen, figs5-per-animal, figs4-lmer-feature-sets | tested (mixed models) |

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"  # many small model fits run in parallel processes
import sys, io, tarfile, tempfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from concurrent.futures import ThreadPoolExecutor
import qu2025 as q

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T_START = time.time()
print("dandiset", q.DANDISET, "version", q.VERSION)

## Loading

Every session's trial table is streamed from the archive; the two photometry channels of each
session that has them are streamed and preprocessed into Z(DA) (Methods, Fiber Photometry: airPLS
trend removal on both channels, a robust regression of 415 nm onto 470 nm, the fitted 415 nm signal
subtracted, z-scored per session; see `qu2025.preprocess`).

In [ ]:
trials, meta = q.load_trials()
d = q.behaviour_frame(trials)
meta["Session"] = meta["subject"] + "_" + meta["session_id"]
print(len(trials), "trials in", len(meta), "files")


def fp_job(row):
    fp = q.load_photometry(row.asset_id)
    if len(fp["t470"]) < 1000:
        return row.Session, None, len(fp["t470"]), fp
    t, z, sig, ref = q.preprocess(fp)
    return row.Session, (t, z, sig, ref), len(fp["t470"]), fp


with ThreadPoolExecutor(4) as ex:
    fp_out = list(ex.map(fp_job, meta[meta["has_fp"]].itertuples()))
Z = {s: r for s, r, n, fp in fp_out if r is not None}
RAW = {s: fp for s, r, n, fp in fp_out}
print("photometry files", len(fp_out), "preprocessed", len(Z), "too short:",
      [(s, n) for s, r, n, fp in fp_out if r is None])
print("load time %.0f s" % (time.time() - T_START))

## Data check

In [ ]:
print("== Subjects and sessions")
print(meta.groupby("subject").agg(sessions=("Session", "size"), with_photometry=("has_fp", "sum"),
                                  first_day=("age", "min"), last_day=("age", "max"),
                                  trials=("n_trials", "sum")))
print("sex:", meta.sex.value_counts().to_dict(), "| species:", meta.species.value_counts().to_dict(),
      "| subjects:", meta.subject.nunique())
print("session_description:", meta.session_description.value_counts().to_dict())
print("\n== Trial table columns and ranges")
for col in ["center_in", "center_out", "side_in", "outcome", "zeroth_side_out", "first_side_out",
            "last_side_out", "start_time", "stop_time", "tmax", "trial_in_block", "block_num",
            "prebswitch_num", "rewarded"]:
    x = trials[col]
    print(f"{col:16s} n={x.notna().sum():6d} missing={x.isna().sum():4d} min={x.min():10.3f} max={x.max():10.3f}")
for col in ["action", "state", "last_side_out_side"]:
    print(col, trials[col].value_counts(dropna=False).to_dict())
print("\nstop_time equals tmax on every trial:", bool((trials["stop_time"] == trials["tmax"]).all()))
print("\n== Task structure (Methods: correct port rewarded with p = 0.75, other never; switch after 7 to 23 rewards)")
v = d[d["valid"]]
print("P(reward | correct choice) = %.4f, P(reward | incorrect) = %.4f" %
      (v[v.correct == 1].Reward.mean(), v[v.correct == 0].Reward.mean()))
blk = d.groupby(["Session", "blockNum"]).agg(nrew=("Reward", "sum"), n=("Trial", "size"),
                                             states=("Target", "nunique"))
last = blk.groupby(level=0).tail(1).index
done = blk.drop(last)
print("rewards per completed block: min %d max %d mean %.2f (n blocks %d); one correct port per block: %s"
      % (done.nrew.min(), done.nrew.max(), done.nrew.mean(), len(done), bool((blk.states == 1).all())))
print("trials with no choice:", int((~d.valid).sum()), "of", len(d))
print("\n== Timing measures (s), Methods definitions")
print(d[["center_dur", "port_dur", "MVMT", "SO_lat"]].describe().T.round(3))

In [ ]:
print("== Photometry files")
fpm = meta[meta["has_fp"]].copy()
fpm["fs_470"] = [1 / np.median(np.diff(RAW[s]["t470"])) if len(RAW[s]["t470"]) > 1 else np.nan
                 for s in fpm.Session]
fpm["first_gap_s"] = [np.diff(RAW[s]["t470"])[:3].max() if len(RAW[s]["t470"]) > 3 else np.nan
                      for s in fpm.Session]
print(fpm[["Session", "location", "coordinates", "unit", "n_470", "n_415", "fs_470", "first_gap_s"]]
      .to_string(index=False))
print("\nLocation per mouse in day order (L = left, R = right, - = no photometry):")
for s, g in meta.sort_values("age").groupby("subject"):
    print(s, "".join("-" if not h else ("L" if l.startswith("left") else "R")
                     for h, l in zip(g.has_fp, g.location.fillna(""))))
print("\nraw value ranges (unit 'F'): 470 nm %.4f to %.4f, 415 nm %.4f to %.4f" % (
    min(RAW[s]["f470"].min() for s in Z), max(RAW[s]["f470"].max() for s in Z),
    min(RAW[s]["f415"].min() for s in Z), max(RAW[s]["f415"].max() for s in Z)))
print("Z(DA) after preprocessing: mean %.2e, sd %.3f (z-scored per session)" %
      (np.mean([Z[s][1].mean() for s in Z]), np.mean([Z[s][1].std() for s in Z])))
print("\n== Duplicate recordings: identical photometry lengths or identical first 1000 samples")
sig = {}
for s in Z:
    key = (len(RAW[s]["f470"]), round(float(RAW[s]["f470"][100:1100].sum()), 6))
    sig.setdefault(key, []).append(s)
print("pairs:", [v for v in sig.values() if len(v) > 1])
tt = d.groupby("Session").apply(lambda g: (len(g), round(g.center_in.sum(), 3)))
print("identical trial tables:", [list(g.index) for _, g in tt.groupby(tt) if len(g) > 1])

### Clock check and sessions with a dopamine signal

The photometry timestamps and the trial events are meant to share the session clock. For each
session, the mean de-based Z(DA) over 0 to 1 s after outcome is compared between rewarded and
unrewarded trials (t statistic) with the true outcome times and with the outcome times shifted by
-3, +3 and +10 s. A response present only at the true times shows that the streams share a clock.

In [ ]:
def contrast(s, shift):
    t, z = Z[s][0], Z[s][1]
    x = d[d.Session == s]
    g, y = q.outcome_aligned(t, z, x.outcome.values + shift, 1.0)
    _, mn = q.da_measures(g, y, x.Reward.values)
    a, b = mn[x.Reward.values == 1], mn[x.Reward.values == 0]
    return stats.ttest_ind(a[~np.isnan(a)], b[~np.isnan(b)]).statistic


chk = pd.DataFrame([{"Session": s, **{f"t_shift_{sh}": contrast(s, sh) for sh in (0, -3, 3, 10)}}
                    for s in Z])
chk["responds"] = chk.t_shift_0 > 5
chk = chk.merge(meta[["Session", "n_trials", "location"]], on="Session")
chk["auc_470_vs_rescaled_415"] = [q.roc_auc(Z[s][2], Z[s][3]) for s in chk.Session]
print(chk.round(3).to_string(index=False))
print("\nsessions with a response at the true outcome times:", int(chk.responds.sum()), "of", len(chk),
      "; trials in them:", int(chk[chk.responds].n_trials.sum()))
print("sessions without:", list(chk[~chk.responds].Session))
print("AUC (470 nm against rescaled 415 nm samples): min %.3f max %.3f; sessions above 0.9: %d"
      % (chk.auc_470_vs_rescaled_415.min(), chk.auc_470_vs_rescaled_415.max(),
         (chk.auc_470_vs_rescaled_415 > 0.9).sum()))

In [ ]:
# for the sessions without a response, search a clock offset of up to +-400 s
def best_shift(s):
    t, z = Z[s][0], Z[s][1]
    x = d[d.Session == s]
    rw = x.Reward.values == 1
    shifts = np.arange(-400, 400, 0.5)
    eff = []
    for sh in shifts:
        g, y = q.outcome_aligned(t, z, x.outcome.values + sh, 0.5)
        m = np.nanmean(y[:, 1:], axis=1)
        eff.append((np.nanmean(m[rw]) - np.nanmean(m[~rw])) / np.nanstd(m))
    eff = np.array(eff)
    i = np.argmax(np.abs(eff))
    return shifts[i], eff[i], eff[np.argmin(np.abs(shifts))]


for s in list(chk[~chk.responds].Session) + list(chk[chk.responds].Session[:2]):
    sh, e, e0 = best_shift(s)
    print(f"{s}: largest standardized reward contrast {e:+.2f} at shift {sh:+.1f} s; at 0 s {e0:+.2f}")

The claims file's reading of the session selection (AUC of 470 nm samples against rescaled 415 nm
samples above 0.9) keeps no session: every AUC is near 0.5, as both detrended channels are centred
on zero. The paper's other criteria (visual inspection, longer tails, no discontinuities) cannot be
redone. Sessions are therefore taken in two sets, and the dopamine claims are run on both:

- `S38`, the main set (assumption): the 38 sessions in which outcome dopamine responds to reward at
  the true event times. The 8 sessions without a response (all of mouse BSD011) show none at any
  clock offset up to 400 s, so they hold no usable dopamine signal and would fail a visual
  inspection.
- `S46`: every session with photometry long enough to process (46; two files of BSD016 hold 30 and
  40 samples).

In [ ]:
S38 = set(chk[chk.responds].Session)
S46 = set(chk.Session)
hem = meta.set_index("Session")["location"].str[0].map({"l": 0, "r": 1})


def neural_frame(sessions):
    rows = []
    for s in sorted(sessions):
        t, z = Z[s][0], Z[s][1]
        x = d[d.Session == s].copy()
        g, y = q.outcome_aligned(t, z, x.outcome.values, 6.0)
        x["DA_PT"], x["DA_mean"] = q.da_measures(g, y, x.Reward.values)
        x["trace"] = list(y)
        rows.append(x)
    n = pd.concat(rows)
    n["hem"] = n.Session.map(hem)
    n["contra"] = ((n.Decision >= 0) & (n.Decision != n.hem)).astype(int)
    n["ipsi"] = 1 - n["contra"]
    n["reward"] = (n.Reward == 1).astype(int)
    n["omission"] = (n.Reward == 0).astype(int)
    n["switch"] = n["Switch"]
    return n


GRID = np.arange(0, 6.0 + q.DT / 2, q.DT)
N46 = neural_frame(S46)
N38 = N46[N46.Session.isin(S38)]
print("trials: S46", len(N46), "S38", len(N38))
print("DA_PT range %.2f to %.2f, DA_mean range %.2f to %.2f (z units); missing DA_PT %d" %
      (N46.DA_PT.min(), N46.DA_PT.max(), N46.DA_mean.min(), N46.DA_mean.max(), N46.DA_PT.isna().sum()))
print("trials per mouse in S38:", N38.groupby("Subject").size().to_dict())

### Model code check

The paper's code is fetched at the pinned commit; its own likelihood function
(`cogmodels.utils.cogmodel_nll`) and its simulated RPEs are compared with `qu2025.run_model` on the
first three sessions of each mouse, at two random parameter sets per model.

In [ ]:
url = f"https://codeload.github.com/Wilbrecht-Lab/cogmodels/tar.gz/{q.COGMODELS_COMMIT}"
tmp = tempfile.mkdtemp()
with tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen(url).read()), mode="r:gz") as tf:
    tf.extractall(tmp)
sys.path.insert(0, os.path.join(tmp, f"cogmodels-{q.COGMODELS_COMMIT}"))
import cogmodels as cm

classes = {"RL4p": cm.RL_4p, "RLCF": cm.RLCF, "RLFQ3p": cm.RL_Forgetting3p, "PearceHall": cm.Pearce_Hall,
           "RL_meta": cm.RL_Grossman, "BIfp": cm.BIModel_fixp, "BRLfwr": cm.BRL_fwr, "BRLfw": cm.BRL_fw,
           "BRLwrp": cm.BRL_wrp, "RFLR": cm.RFLR}
rng = np.random.default_rng(11)
cols = ["ID", "Subject", "Session", "Trial", "blockTrial", "Target", "Decision", "Reward"]
rows = []
for subj in sorted(d.Subject.unique()):
    sub = d[(d.Subject == subj) & (d.session_num <= 3)].reset_index(drop=True)
    for name, cls in classes.items():
        for k in range(2):
            x = q.init_params(name, rng)
            m = cls()
            data = sub[cols].copy()
            data["Reward"] = data["Reward"].fillna(0).astype(int)
            if name != "RFLR":
                data = m.fit_marginal(data)
            params = pd.DataFrame([dict(zip(q.MODELS[name][1], x), ID=subj)])
            ref = cm.utils.cogmodel_nll(m, data.copy(), params)
            mine, _, rp = q.model_nll(name, x, *q.model_arrays(sub))
            sim = m.sim(data.copy(), params)
            rd = np.nanmax(np.abs(sim["rpe"].values - rp)) if "rpe" in sim else np.nan
            rows.append((subj, name, ref, mine, rd))
cc = pd.DataFrame(rows, columns=["subject", "model", "nll_code", "nll_here", "max_rpe_diff"])
print("largest |nll difference|: %.2e over %d comparisons; largest RPE difference %.2e"
      % ((cc.nll_code - cc.nll_here).abs().max(), len(cc), cc.max_rpe_diff.max()))
print(cc.groupby("model")[["nll_code", "nll_here"]].mean().round(4))

## data-composition

> The data are choice behaviour of 5 male mice over 14 full-task sessions each, with dLight
> photometry from nucleus accumbens core in a left, right, no-recording sequence.

In [ ]:
spm = meta.groupby("subject").agg(sessions=("Session", "size"), photometry_files=("has_fp", "sum"))
spm["photometry_usable"] = [sum(s in S46 for s in meta[meta.subject == m].Session) for m in spm.index]
print(spm)
print("mice:", meta.subject.nunique(), "| sessions:", len(meta), "| sessions per mouse:",
      sorted(spm.sessions.unique()))
print("fraction of sessions with a photometry file: %.3f (expected about 2/3 = 0.667)" % meta.has_fp.mean())
print("sex:", meta.groupby("subject").sex.first().to_dict())
print("BSD017 days:", sorted(meta[meta.subject == "BSD017"].age.str[1:-1].astype(int)))

**Result: partially reproduced.** The dataset holds 5 mice but 69 sessions: BSD017 has 13 (days
153 to 164 and 167; days 165 and 166 are absent). 48 of 69 sessions (0.696) have a photometry
file, close to two of three, but BSD011 was recorded in all 14 sessions, with no no-recording day,
and two BSD016 files hold almost no samples.

## perf-after-pretraining

> Just after pretraining mice chose the high-probability port 72.9% of the time on average (95% CI
> 0.669 to 0.767).

In [ ]:
rng = np.random.default_rng(101)
v = d[d.valid]
rows = []
for lab, sel in [("(a) first session", v.session_num == 1), ("(b) all sessions", v.session_num > 0)]:
    x = v[sel]
    mice = x.groupby("Subject").correct.mean()
    rows.append((lab, "(i) pooled", x.correct.mean(), *q.bootstrap_ci(x.correct.values, rng)))
    rows.append((lab, "(ii) mean of mice", mice.mean(), *q.bootstrap_ci(mice.values, rng)))
    print(lab, "per mouse:", mice.round(3).to_dict())
print(pd.DataFrame(rows, columns=["sessions", "average", "fraction correct", "ci_low", "ci_high"]).round(4))
print("chance level 0.5; paper 0.729 [0.669, 0.767]")

**Result: reproduced.** In the first full-task session mice chose the correct port on 0.737 of
trials pooled and 0.732 as the mean of the mice, both inside [0.669, 0.767]; over all sessions the
values are 0.769 and 0.768, just above the interval.

## fig1f-performance-increase

> Over the 14 sessions performance rose steadily, with a positive linear regression slope of
> 0.0026 per session (t = 2.743, p = 0.008).

In [ ]:
ps = v.groupby(["Subject", "session_num"]).correct.mean().reset_index()
m1 = smf.ols("correct ~ session_num", ps).fit()
m2 = smf.ols("correct ~ session_num", v).fit()
for lab, m in [("(a) per mouse and session", m1), ("(b) per trial", m2)]:
    ci = m.conf_int().loc["session_num"].values
    print(f"{lab}: n = {int(m.nobs)}, slope {m.params.session_num:.5f} [{ci[0]:.5f}, {ci[1]:.5f}], "
          f"t = {m.tvalues.session_num:.3f}, p = {m.pvalues.session_num:.2g}, df = {int(m.df_resid)}")
print("per mouse slopes:", {s: round(smf.ols("correct ~ session_num", g).fit().params.session_num, 4)
                            for s, g in ps.groupby("Subject")})
fig, ax = plt.subplots(figsize=(4, 3))
for s, g in ps.groupby("Subject"):
    ax.plot(g.session_num, g.correct, "o-", ms=3, label=s)
ax.set_xlabel("session"); ax.set_ylabel("fraction correct"); ax.legend(fontsize=6)
print("plotted range: %.3f to %.3f" % (ps.correct.min(), ps.correct.max()))

**Result: reproduced.** Over the 69 mouse-session values the slope is 0.0027 per session
(95% CI 0.0008 to 0.0046), t = 2.86, p = 0.0057, inside [0.001, 0.005]; the per-trial reading gives
0.0024 (p = 7.6e-08).

## trials-to-switch, trials-to-commit, figs1-switch-measures-decrease

> After a block switch mice took 2.49 trials on average to switch to the new correct port in the
> first 7 sessions, and 2.36 trials in sessions 8 to 14.

> Mice switched to and committed to the correct port for 3 or more trials after an average of 3.12
> trials (95% CI 3.00 to 3.26), following the first 7 sessions.

> The number of trials to switch and to commit to the correct port fell over blocks within a
> session and over sessions.

In [ ]:
bs = q.block_switch_measures(d)
rng = np.random.default_rng(102)
print("block switches:", len(bs), "| without a switch to the new port:", int(bs.trial2sw.isna().sum()),
      "| without 3 consecutive correct:", int(bs.trial2asymp.isna().sum()))
rows = []
for col in ["trial2sw", "trial2asymp"]:
    for lab, sel in [("1-7", bs.session_num <= 7), ("8-14", bs.session_num >= 8)]:
        x = bs[sel].dropna(subset=[col])
        for cnt, off in [("(b) first trial = 0", 0), ("(a) first trial = 1", 1)]:
            pooled = x[col] + off
            mice = x.groupby("Subject")[col].mean() + off
            rows.append((col, lab, cnt, "pooled", pooled.mean(), *q.bootstrap_ci(pooled.values, rng), len(x)))
            rows.append((col, lab, cnt, "mean of mice", mice.mean(), *q.bootstrap_ci(mice.values, rng), 5))
res_sw = pd.DataFrame(rows, columns=["measure", "sessions", "count", "average", "value", "ci_low", "ci_high", "n"])
print(res_sw.round(3).to_string(index=False))
print("\nper mouse, count from 0:")
print(bs.groupby(["Subject", bs.session_num >= 8])[["trial2sw", "trial2asymp"]].mean().round(2))

**Result (trials-to-switch): reproduced.** Counting the first trial of the new block as 0, the
pooled mean is 2.46 in sessions 1 to 7 and 2.33 in sessions 8 to 14, inside [2.39, 2.59] and
[2.28, 2.45]; the mean of the mice gives 2.47 and 2.33. Counting from 1, both values are 1 higher
and outside.

**Result (trials-to-commit): reproduced.** In sessions 8 to 14, counting from 0, the pooled mean is
3.03 (mean of mice 3.08), inside [3.00, 3.26]; in sessions 1 to 7 it is 3.47.

In [ ]:
bs["block_in_session"] = bs.blockNum - 1
rows = []
for col in ["trial2sw", "trial2asymp"]:
    for xv in ["block_in_session", "session_num"]:
        m = smf.ols(f"{col} ~ {xv}", bs.dropna(subset=[col])).fit()
        rows.append((col, xv, m.params[xv], m.pvalues[xv], int(m.nobs)))
print(pd.DataFrame(rows, columns=["measure", "on", "slope", "p", "n"]).to_string(index=False))
print("block number range 1 to", int(bs.block_in_session.max()))

**Result: reproduced.** All four OLS slopes are negative: trial2sw on block number -0.0076
(p = 0.0097) and on session -0.024 (p = 0.0027); trial2asymp on block number -0.0036 (p = 0.48) and
on session -0.081 (p = 2.2e-09). The within-session slope of trial2asymp is not significant.

## fig1e-stay-after-omission

> The probability of repeating the last choice fell after one unrewarded outcome (RR against RU)
> and fell further after two (RU against UU), with Fisher's exact test statistics of 6.47 and 3.98.

In [ ]:
c = d.Decision
same2 = d.Session.eq(d.Session.shift(2))
sel = (d.Trial > 2) & same2 & (c >= 0) & (c.shift(1) >= 0) & (c.shift(2) >= 0) & (c.shift(1) == c.shift(2))
hist2 = (np.where(d.Reward.shift(2) == 1, "R", "U").astype(object)
         + np.where(d.Reward.shift(1) == 1, "R", "U").astype(object))
h2 = pd.DataFrame({"h": hist2[sel], "stay": (c == c.shift(1))[sel].astype(int), "Subject": d.Subject[sel]})
print("history = outcome at t-2, then t-1; trials t with the same port at t-2 and t-1")
print(h2.groupby("h").stay.agg(["size", "mean"]).round(4))
print("paper's counts as printed: RR 23607, RU 5719, UR 9428, 'RU' (read UU) 8520")


def compare(a, b):
    A = h2[h2.h == a].stay
    B = h2[h2.h == b].stay
    tab = [[A.sum(), len(A) - A.sum()], [B.sum(), len(B) - B.sum()]]
    o, p = stats.fisher_exact(tab)
    sp = np.sqrt(((len(A) - 1) * A.var() + (len(B) - 1) * B.var()) / (len(A) + len(B) - 2))
    return o, p, (A.mean() - B.mean()) / sp


for a, b in [("RR", "RU"), ("RU", "UU")]:
    o, p, dd = compare(a, b)
    print(f"{a} vs {b}: odds ratio {o:.3f}, Fisher p = {p:.1e}, Cohen's d {dd:.4f}")
print("\nper mouse stay rate:")
print(h2.pivot_table(index="Subject", columns="h", values="stay").round(3))

**Result: reproduced.** Stay rates are 0.961 (RR), 0.791 (RU) and 0.485 (UU). RR against RU gives
an odds ratio of 6.50 and Cohen's d 0.625; RU against UU 4.02 and 0.675; both Fisher p values are printed as 0 (below the floating-point range). All
four are within tolerance. The counts of RR (23302) and UU (8429) are within 10% of the paper's;
the paper's RU and UR counts are reversed with respect to these (here RU, reward then omission,
9297, and UR 5649, against the paper's RU 5719 and UR 9428), so the paper's labels for those two
counts name the outcomes in the other order.

## figs6a-decaying-history-weights

> Logistic regressions fitted to each mouse show weights of past choice-outcome history that decay
> for more distant trials, consistently across mice.

The paper code's `LR` class (4 lags, binomial GLM) is run on each mouse's trials.

In [ ]:
lrw = {}
for s, g in d.groupby("Subject"):
    m = cm.LR(nlag=4)
    m.fitsim(g[["ID", "Session", "Trial", "Decision", "Reward"]].reset_index(drop=True).copy())
    lrw[s] = m.fitted_params.iloc[0].drop("ID").astype(float)
lrw = pd.DataFrame(lrw)
print(lrw.round(3))
cr = lrw.loc[[f"C_{i}back:R_{i}back" for i in range(1, 5)]]
ok = (cr.iloc[0] > 0) & (cr.iloc[0] == cr.max())
print("\nlag-1 choice x reward positive and largest:", ok.to_dict())
print("mean over mice by lag:", cr.mean(axis=1).round(3).to_dict())
print("monotone decrease of the mean:", bool(np.all(np.diff(cr.mean(axis=1).values) < 0)))

**Result: reproduced.** In all 5 mice the lag-1 choice-by-reward weight is positive and the largest
(1.97 to 2.91), and the mean over mice falls from 2.40 at lag 1 to 0.88, 0.36 and 0.08 at lags 2
to 4.

## Model fits

Each model is fitted per mouse to all its sessions (code, `fit_model_per_subject`; L-BFGS-B, the
code's bounds, one start drawn from the code's initial distributions, seed 230, a standard choice).
RLFQ3p is fitted with the code's forgetting rule (zeta = 1 - (alpha+ + alpha-)/2) and with the
Methods' rule (zeta = (alpha+ + alpha-)/2, `RLFQ3p_methods`).

In [ ]:
FIT_MODELS = ["RL4p", "RLCF", "RLFQ3p", "RLFQ3p_methods", "PearceHall", "RL_meta", "BIfp", "BRLfwr",
              "BRLfw", "BRLwrp", "RFLR"]
fits = pd.DataFrame([dict(q.fit_model(m, g.reset_index(drop=True)), Subject=s)
                     for m in FIT_MODELS for s, g in d.groupby("Subject")])
print("fits:", len(fits), "| optimiser reported success:", int(fits.success.sum()))
aic = fits.pivot(index="Subject", columns="model", values="aic")
rel = aic.sub(aic["RL4p"], axis=0)
print("AIC relative to RL4p, per mouse:")
print(rel.round(2).T)

In [ ]:
def ttests(diff_cols, family):
    # t-test of per-mouse differences against 0, Holm-corrected within the family
    rows = []
    for col in family:
        x = diff_cols[col]
        t, p2 = stats.ttest_1samp(x, 0)
        p1 = p2 / 2 if x.mean() < 0 else 1 - p2 / 2
        rows.append((col, x.mean(), x.std(ddof=1), x.std(ddof=1) / np.sqrt(len(x)), t, p2, p1))
    r = pd.DataFrame(rows, columns=["model", "mean", "sd", "sem", "t", "p_two", "p_one"]).set_index("model")
    r["p_two_holm"] = q.holm(r.p_two.values)
    r["p_one_holm"] = q.holm(r.p_one.values)
    return r


FAM_A = ["BRLfwr", "BIfp", "RFLR", "RLFQ3p", "RL_meta", "PearceHall"]
FAM_B = FAM_A + ["RLCF"]
for fam_name, fam in [("(a) six models", FAM_A), ("(b) with RLCF", FAM_B)]:
    print(fam_name)
    print(ttests(rel, fam).round(4))
    print()
print("RLFQ3p with the Methods' forgetting rule:", ttests(rel, ["RLFQ3p_methods"]).round(4).to_dict("records"))

**Result (fig2c-bayesian-beat-rl4p): reproduced.** The mean AIC relative to RL4p is -535.36 for
BIfp and -530.09 for BRLfwr (paper -536.57 and -530.59), and every Holm-corrected p is below 0.05
in both families, two-sided and one-sided.

**Result (fig2c-complex-rl-beat-rl4p): reproduced.** RFLR -648.75, RL_meta -679.73, PearceHall
-678.01 and RLFQ3p -645.17 (paper -652.53, -677.84, -656.44, -646.45), all within the paper's
spreads, with Holm-corrected p below 0.05 under every family and side. With the Methods'
forgetting rule RLFQ3p gives -460.66, outside the tolerance; the code's rule matches the paper.

In [ ]:
print("RLCF relative to RL4p: %.2f (sd %.2f); RLCF - BIfp: %.2f (sd %.2f)" % (
    rel.RLCF.mean(), rel.RLCF.std(), (aic.RLCF - aic.BIfp).mean(), (aic.RLCF - aic.BIfp).std()))
fam5 = ["RL_meta", "PearceHall", "RFLR", "RLFQ3p", "RLCF"]
bt = ttests(-(aic[fam5].sub(aic["BRLfwr"], axis=0)), fam5)  # AIC(BRLfwr) - AIC(model); p_one: BRLfwr lower
print("AIC(BRLfwr) - AIC(model), t-tests within the five BRLfwr comparisons:")
print(bt.round(4))

**Result (fig2c-rlcf): reproduced.** RLCF is -364.50 relative to RL4p (paper -361.92) and 170.87
above BIfp (paper 174.64). BRLfwr has a lower AIC than RLCF in every mouse (mean difference
-165.59), with Holm-corrected p = 0.0097 two-sided and 0.0049 one-sided.

**Result (fig2c-no-bayes-vs-complex-rl): reproduced.** With two-sided tests and Holm correction over
the five comparisons, BRLfwr does not differ from RL_meta, PearceHall, RFLR or RLFQ3p (corrected p
0.201 to 0.214); the complex RL models have the lower AIC on average (by 115 to 150). One-sided, in
the direction BRLfwr better, none is significant either (corrected p 1.0).

In [ ]:
r4 = fits[fits.model == "RL4p"].set_index("Subject")[["beta", "a_pos", "a_neg", "st"]]
print(r4.round(4))
rng = np.random.default_rng(103)
for p in ["a_pos", "a_neg"]:
    print(f"(a) per-mouse fits: mean {p} = {r4[p].mean():.4f}, bootstrap CI {np.round(q.bootstrap_ci(r4[p].values, rng), 3)}")
sess_fits = pd.DataFrame([dict(q.fit_model(m, g.reset_index(drop=True)), Subject=g.Subject.iat[0],
                               Session=s, session_num=g.session_num.iat[0], n_choice=int(g.valid.sum()))
                          for m in ["RL4p", "BIfp", "BRLfwr"] for s, g in d.groupby("Session")])
s4 = sess_fits[sess_fits.model == "RL4p"]
for p in ["a_pos", "a_neg"]:
    print(f"(b) per-session fits ({len(s4)}): mean {p} = {s4[p].mean():.4f}, CI {np.round(q.bootstrap_ci(s4[p].values, rng), 3)}")

**Result: reproduced.** Fitted per mouse, RL4p has mean alpha+ 0.848 and alpha- 0.731, inside
[0.77, 0.94] and [0.71, 0.75] (paper 0.85 and 0.73). Per-session fits (reading b) give 0.800 and
0.738, also inside.

In [ ]:
bw = fits[fits.model == "BRLfw"].set_index("Subject")["alpha"]
print("BRLfw alpha per mouse:", bw.round(6).to_dict())
print("mice with alpha < 1e-6:", int((bw < 1e-6).sum()), "of 5")
print("AIC(BRLfw) - AIC(BIfp) per mouse:", (aic.BRLfw - aic.BIfp).round(3).to_dict())

**Result: reproduced.** The fitted alpha of BRLfw is 0 (at the bound) in 4 of 5 mice and 0.029 in
BSD015; with alpha 0 the model's AIC is BIfp's plus 2 for the extra parameter.

In [ ]:
print("figs7: per-session fits of RL4p, BIfp and BRLfwr (behaviour)")
sf = sess_fits.pivot_table(index=["Subject", "Session", "session_num", "n_choice"], columns="model",
                           values="aic").reset_index()
rows = []
for m in ["BIfp", "BRLfwr"]:
    sf[f"rel_{m}"] = sf[m] - sf["RL4p"]
    sf[f"per_trial_{m}"] = sf[m] / sf["n_choice"]
    for lab, col in [("(a) AIC relative to RL4p", f"rel_{m}"), ("(b) AIC per choice", f"per_trial_{m}")]:
        r = smf.ols(f"{col} ~ session_num", sf).fit()
        rows.append((m, lab, r.params.session_num, r.pvalues.session_num, int(r.nobs)))
print(pd.DataFrame(rows, columns=["model", "reading", "slope per session", "p", "n"]).to_string(index=False))
print("\nper-mouse slopes of relative AIC:")
print(sf.groupby("Subject").apply(lambda g: pd.Series({m: np.polyfit(g.session_num, g[f"rel_{m}"], 1)[0]
                                                       for m in ["BIfp", "BRLfwr"]})).round(3))

**Result: not reproduced.** Fitted to single sessions, BIfp and BRLfwr improve over sessions under
both readings of model fitness: relative to RL4p their AIC falls by 1.94 and 1.96 per session
(p = 0.0068 and 0.0038), and per choice by 0.0122 per session (p = 0.0077 and 0.0074). The
relative-AIC slope is negative in each of the 5 mice for both models. The paper names no test; by
the claim's OLS the Bayesian models fit better in later sessions.

## fig2b-model-identifiability and figs2-parameter-recovery

> In model identification, the Bayesian model and the hybrid BRL are poorly told apart, and so are
> the complex RL models (RL_meta, RLFQ3p, RFLR), but the families can be told apart and the
> standard RL model RL4p can be rejected.

> Parameters are recovered well for the Bayesian models such as BRLfwr, while RL_meta, the most
> complex model with 7 parameters, has poor parameter identifiability.

Two simulation readings (claim): (b) the code's, 500 sessions of 500 trials per model on blocks of 8
to 15 rewards; (a) the Methods', 1000 per model on blocks of 7 to 23 rewards, with the code's 500
trials, as the Methods give no length. Each simulated session is fitted with all 8 models.

In [ ]:
ID_MODELS = ["RL4p", "RLCF", "RFLR", "BRLfwr", "BIfp", "RLFQ3p", "RL_meta", "PearceHall"]
q._idjob(("RL4p", [2, 0.8, 0.7, 0.5], 500, (8, 15), 1, ID_MODELS))  # compile before forking
t0 = time.time()
ident = {"(b) code": q.identification(fits, ID_MODELS, 500, 500, (8, 15), 230),
         "(a) Methods": q.identification(fits, ID_MODELS, 1000, 500, (7, 23), 231)}
print("simulation and refitting time %.0f s" % (time.time() - t0))
FAM = {"BIfp": "Bayes", "BRLfwr": "Bayes", "RL_meta": "cRL", "RLFQ3p": "cRL", "RFLR": "cRL",
       "PearceHall": "cRL", "RL4p": "RL4p", "RLCF": "RLCF"}


def confusion(R):
    best = R.loc[R.groupby(["sim", "rep"]).aic.idxmin()]
    return pd.crosstab(best.sim, best.model, normalize="index").reindex(index=ID_MODELS, columns=ID_MODELS).fillna(0)


def judge(C):
    nonb = [m for m in ID_MODELS if FAM[m] != "Bayes"]
    c1 = (C.loc["BIfp", "BRLfwr"] > C.loc["BIfp", nonb].max() and C.loc["BRLfwr", "BIfp"] > C.loc["BRLfwr", nonb].max()
          and min(C.loc["BIfp", "BIfp"], C.loc["BRLfwr", "BRLfwr"]) < 0.5)
    grp = ["RL_meta", "RLFQ3p", "RFLR"]
    out = [m for m in ID_MODELS if m not in grp]
    c2 = all(C.loc[g, [x for x in grp if x != g]].sum() > C.loc[g, out].max() for g in grp) and \
        min(C.loc[g, g] for g in grp) < 0.5
    own = {m: C.loc[m, [x for x in ID_MODELS if FAM[x] == FAM[m]]].sum() for m in ID_MODELS}
    c3 = all(v > 0.5 for v in own.values())
    c4 = C.loc["RL4p"].idxmax() == "RL4p" and C.loc[[m for m in ID_MODELS if m != "RL4p"], "RL4p"].max() <= 0.2
    return c1, c2, c3, c4, own


for lab, R in ident.items():
    C = confusion(R)
    print(lab, "- fits per model:", R.groupby("model").size().iloc[0], "; P(best fit j | simulated i):")
    print(C.round(3))
    c1, c2, c3, c4, own = judge(C)
    print("(1) Bayesian pair confused:", c1, "| (2) complex RL confused:", c2,
          "| (3) own family > 0.5 in every row:", c3, "| (4) RL4p identified:", c4)
    print("own-family probability:", {k: round(v, 3) for k, v in own.items()})
    print()

**Result: reproduced.** Under both readings all four parts hold. BRLfwr sessions are best fitted by
BIfp about as often as by BRLfwr (0.46 against 0.40 in the code's reading), RL_meta and PearceHall
sessions are mostly taken by RLFQ3p and RFLR, every row puts more than half its probability on its
own family, and RL4p is identified (0.84 and 0.83) and no other model's sessions go to RL4p more
than 0.054 of the time.

In [ ]:
def recovery(R):
    D = R[R.sim == R.model]
    rows = []
    for m, g in D.groupby("model"):
        for i, p in enumerate(q.MODELS[m][1]):
            tr = np.array([t[i] for t in g.true])
            fi = g[p].values
            mu, sd = tr.mean(), tr.std()
            k = (fi <= mu + 6 * sd) & (fi >= mu - 6 * sd)  # code, viz_genrec_recovery: 6 SD of the true values
            r = np.corrcoef(tr[k], fi[k])[0, 1] if k.sum() > 2 else np.nan
            rows.append((m, p, r, int((~k).sum()), len(k), tr.min(), tr.max()))
    return pd.DataFrame(rows, columns=["model", "param", "r", "dropped", "n", "true_min", "true_max"])


for lab, R in ident.items():
    rec = recovery(R)
    print(lab)
    print(rec.round(3).to_string(index=False))
    good = lambda m: bool((rec[rec.model == m].r >= 0.7).all())
    poor = lambda m: bool((rec[rec.model == m].r < 0.5).any() or rec[rec.model == m].r.isna().any())
    print("BRLfwr good (all r >= 0.7):", good("BRLfwr"), "| RL_meta poor (some r < 0.5):", poor("RL_meta"),
          "| BIfp good:", good("BIfp"))
    print()

**Result: partially reproduced.** RL_meta recovers poorly (psi and alpha_nu correlations near 0,
and its stickiness range collapses so that most fitted values are dropped by the 6 SD rule), but
BRLfwr does not recover well either: only its stickiness reaches r >= 0.7, and its beta, q and alpha
correlations lie between 0.3 and 0.65 under both readings. The parameter ranges fitted over the 5
mice are narrow for several parameters (printed as true_min and true_max), which limits these
correlations.

## fig2f-adaptation-after-switch

> BRLfwr and RLCF adapt to block switches faster than RL4p, resembling the mice.

In [ ]:
rng_seed = 104
FPAR = {(r.model, r.Subject): np.array([getattr(r, p) for p in q.MODELS[r.model][1]]) for r in fits.itertuples()}


def pcorrect_curve(df, choices=None):
    x = df.copy()
    if choices is not None:
        x["Decision"] = choices
    x = x[(x.Decision >= 0) & (x.blockNum > 1) & (x.blockTrial <= 10)] if "blockNum" in x else \
        x[(x.Decision >= 0) & (x.blockTrial <= 10)]
    return x.groupby("blockTrial").apply(lambda g: (g.Decision == g.Target).mean())


curves = {"mice": pcorrect_curve(d)}
for m in ["RL4p", "BRLfwr", "RLCF"]:
    em = []
    gen = {"8-15": [], "7-23": []}
    for k, (s, g) in enumerate(d.groupby("Subject")):
        g = g.reset_index(drop=True)
        C, R, T, BT, T1, P = q.emulate_frame(m, FPAR[(m, s)], g, rng_seed + k)
        em.append(pcorrect_curve(g, C).rename(s))
        for lab, blocks in [("8-15", (8, 15)), ("7-23", (7, 23))]:
            sim, _ = q.generate_frame(m, FPAR[(m, s)], int(g.groupby("Session").size().mean()),
                                      g.Session.nunique(), blocks, rng_seed + 10 + k)
            sim["blockNum"] = (sim.groupby("Session").blockTrial.transform(lambda b: (b == 0).cumsum()))
            gen[lab].append(pcorrect_curve(sim).rename(s))
    curves[f"{m} emulate"] = pd.concat(em, axis=1).mean(axis=1)
    for lab in gen:
        curves[f"{m} generate {lab}"] = pd.concat(gen[lab], axis=1).mean(axis=1)
cv = pd.DataFrame(curves)
print(cv.round(3).T)
mean = cv.mean()
print("\nmean P(correct) over block trials 0 to 10:")
print(mean.round(4).to_string())
for rd in ["emulate", "generate 8-15", "generate 7-23"]:
    r4, br, rc = mean[f"RL4p {rd}"], mean[f"BRLfwr {rd}"], mean[f"RLCF {rd}"]
    mm = mean["mice"]
    print(f"{rd}: BRLfwr > RL4p {br > r4}, RLCF > RL4p {rc > r4}, |BRLfwr - mice| {abs(br - mm):.3f}, "
          f"|RLCF - mice| {abs(rc - mm):.3f}, |RL4p - mice| {abs(r4 - mm):.3f}")

**Result: reproduced.** Under all three readings, BRLfwr and RLCF are correct more often than RL4p
over block trials 0 to 10, and closer to the mice. Emulated on the mice's sessions: mice 0.702,
BRLfwr 0.687, RLCF 0.681, RL4p 0.617. Generated with blocks of 8 to 15 rewards: 0.667, 0.667 and
0.597; with 7 to 23: 0.675, 0.667 and 0.589.

## fig3b-sse and fig3a-both-port-contexts

> Against mouse switch probabilities after each three-trial history, the models' errors are RL4p
> 0.8270, RLCF 0.2748, BRLfwr 0.1796, RFLR 0.1537, PearceHall 0.1084 and RL_meta 0.0787; RL4p fits
> worst.

> After histories with rewards at both ports in the last 3 trials, mice rarely switched (0.2143) and
> BRLfwr predicted a similar rate (0.1801), while RL4p overestimated it (0.4328); RLCF gave 0.2248
> and RFLR 0.2778.

In [ ]:
H_MODELS = ["RL4p", "RLCF", "BRLfwr", "RFLR", "PearceHall", "RL_meta"]
d["hist3"] = q.three_trial_history(d)
d["sw"] = np.where(d.Decision >= 0, (d.Decision != d.Decision.shift(1)).astype(float), np.nan)
hd = d[d.hist3.notna() & (d.Decision >= 0)].copy()
# (i) model predicted P(switch) on each mouse trial; (ii) emulated choices
for m in H_MODELS:
    pr = np.zeros(len(d))
    emu = []
    for k, (s, g) in enumerate(d.groupby("Subject")):
        gg = g.reset_index(drop=True)
        p, _ = q.model_latents(m, FPAR[(m, s)], gg)
        pr[g.index] = p
        C, R, T, BT, T1, P = q.emulate_frame(m, FPAR[(m, s)], gg, 300 + k)
        e = gg.copy()
        e["Decision"], e["Reward"] = C, R
        e["hist3"] = q.three_trial_history(e)
        e["sw"] = (e.Decision != e.Decision.shift(1)).astype(float)
        emu.append(e[e.hist3.notna()])
    prev = d.Decision.shift(1).values
    d[f"psw_{m}"] = np.where(prev == 1, 1 - pr, pr)
    hd[f"psw_{m}"] = d.loc[hd.index, f"psw_{m}"]
    E = pd.concat(emu)
    hd.attrs[m] = E.groupby("hist3").sw.mean()
mouse_pooled = hd.groupby("hist3").sw.mean()
mouse_mice = hd.groupby(["Subject", "hist3"]).sw.mean().groupby("hist3").mean()
counts = hd.hist3.value_counts()
print("histories with trials:", len(counts), "| trials per history: min", counts.min(), "max", counts.max())
rows = []
for m in H_MODELS:
    mi = hd.groupby("hist3")[f"psw_{m}"].mean()
    mii = hd.attrs[m].reindex(mouse_pooled.index)
    for ml, mv in [("(a) pooled", mouse_pooled), ("(b) mean of mice", mouse_mice)]:
        for pl, pv in [("(i) predicted", mi), ("(ii) emulated", mii)]:
            rows.append((m, ml, pl, np.nansum((mv - pv) ** 2)))
sse = pd.DataFrame(rows, columns=["model", "mouse value", "model value", "SSE"])
print(sse.pivot_table(index="model", columns=["mouse value", "model value"], values="SSE").reindex(H_MODELS).round(4))

**Result: partially reproduced.** RL4p has the largest error under every reading. With the mouse
values pooled over trials and the models' predicted switch probabilities (reading a-i), five of the
six errors are within tolerance: RL4p 0.845, RLCF 0.284, BRLfwr 0.189, RFLR 0.158 and RL_meta
0.082; PearceHall gives 0.081, below the paper's 0.1084 by more than its tolerance. No reading puts
all six within tolerance. 30 of the 32 possible histories have trials.

In [ ]:
both = hd.hist3.map(lambda h: any(ch == "A" for ch in h) and any(ch == "B" for ch in h))
B = hd[both]
print("contexts with rewards at both ports:", sorted(B.hist3.unique()), "| trials", len(B))
rows = []
for lab, val in [("mice", "sw")] + [(m, f"psw_{m}") for m in H_MODELS]:
    pooled = B[val].mean()
    over_ctx = B.groupby("hist3")[val].mean().mean()
    mice = B.groupby(["Subject"])[val].mean().mean()
    rows.append((lab, "(i) predicted" if lab != "mice" else "-", pooled, over_ctx, mice))
for m in H_MODELS:
    E = hd.attrs[m]
    ctx = [h for h in E.index if any(ch == "A" for ch in h) and any(ch == "B" for ch in h)]
    rows.append((m, "(ii) emulated", np.nan, E[ctx].mean(), np.nan))
print(pd.DataFrame(rows, columns=["who", "model value", "(a) pooled", "(b) mean over contexts",
                                  "(c) mean of mice"]).round(4).to_string(index=False))
print("paper: mice 0.2143, BRLfwr 0.1801, RL4p 0.4328, RLCF 0.2248, RFLR 0.2778")

**Result: reproduced.** Averaged over the seven contexts with rewards at both ports (reading b),
with the models' predicted switch probabilities, the mice switch at 0.210, BRLfwr predicts 0.181,
RL4p 0.430, RLCF 0.223 and RFLR 0.276, all within tolerance of the paper's values. Pooled over the
685 trials the mice switch less (0.117).

## fig3c-aic-around-switch

> Around block switches (first 5 trials after a switch) BRLfwr and the dynamic RL models fit
> similarly better than RL4p, with relative AIC of -163.68 (BRLfwr), -163.92 (RL_meta) and -158.91
> (PearceHall).

In [ ]:
ll = {}
for m in ["RL4p", "BRLfwr", "RL_meta", "PearceHall"]:
    lp = np.zeros(len(d))
    for s, g in d.groupby("Subject"):
        p, _ = q.model_latents(m, FPAR[(m, s)], g.reset_index(drop=True))
        p = np.clip(p, 1e-15, 1 - 1e-15)
        c_ = g.Decision.values
        lp[g.index] = np.where(c_ == 1, np.log(p), np.where(c_ == 0, np.log(1 - p), 0.0))
    ll[m] = lp
rows = []
for wl, lo, hi in [("block trials 0-4", 0, 4), ("block trials 1-5", 1, 5)]:
    sel = (d.blockNum > 1) & (d.blockTrial >= lo) & (d.blockTrial <= hi) & d.valid
    for m in ["BRLfwr", "RL_meta", "PearceHall"]:
        k_m, k_0 = len(q.MODELS[m][1]), 4
        per = pd.DataFrame({"s": d.Subject[sel], "m": ll[m][sel], "b": ll["RL4p"][sel]}).groupby("s").sum()
        r_nll = -2 * per.m + 2 * per.b
        r_aic = r_nll + 2 * (k_m - k_0)
        for kl, rr in [("2k - 2 log L", r_aic), ("-2 log L", r_nll)]:
            rows.append((wl, m, kl, rr.mean(), rr.sum()))
r3c = pd.DataFrame(rows, columns=["window", "model", "measure", "mean of mice", "sum over mice"])
print(r3c.round(2).to_string(index=False))
print("trials in window 0-4:", int(((d.blockNum > 1) & (d.blockTrial <= 4) & d.valid).sum()))

**Result: reproduced.** Over block trials 0 to 4 after each switch, with 2k - 2 log L and the mean
over mice, the AIC relative to RL4p is -165.42 for BRLfwr, -164.59 for RL_meta and -166.20 for
PearceHall, all within tolerance of -163.68, -163.92 and -158.91. Over block trials 1 to 5 the
values are -204 to -208, outside.

## trial-cleaning-counts and reward-switch-reward-count

> After trial cleaning 20,952 trials (about 68% of the data) remain across 34 sessions with valid
> dopamine recordings.

> Only 94 trials start a reward, switch, reward sequence, out of 30,707 trials.

In [ ]:
rows = []
for sl, N in [("S38", N38), ("S46", N46)]:
    for so in (1.0, 6.0):
        cN = q.cleaned(N, so)
        rows.append((sl, N.Session.nunique(), so, len(N), len(cN), len(cN) / len(N)))
print(pd.DataFrame(rows, columns=["sessions", "n sessions", "SO_lat limit", "trials", "cleaned", "fraction"])
      .round(3).to_string(index=False))
c1 = q.cleaned(N38, 1.0)
print("\nS38, SO_lat <= 1, trials removed by each rule alone:",
      {k: int(v) for k, v in {"center_dur>0.8": (N38.center_dur > 0.8).sum(), "port_dur>6": (N38.port_dur > 6).sum(),
                              "MVMT>3": (N38.MVMT > 3).sum(), "SO_lat>1": (N38.SO_lat > 1).sum(),
                              "no choice": (~N38.valid).sum(), "missing DA": N38.DA_PT.isna().sum()}.items()})
print("cleaned trials per mouse:", c1.groupby("Subject").size().to_dict())

**Result: partially reproduced, resting on an assumption (the session set).** In the 38 sessions
with a dopamine response, cleaning with
SO_lat <= 1 s leaves 22,370 of 33,415 trials (0.669), within tolerance of 20,952 (about 68%), but
from 38 sessions, not 34. The 6 s reading leaves 30,152, outside. With all 46 sessions the 1 s
reading leaves 25,585, outside.

In [ ]:
def rsr(x):
    R_, c_ = x.Reward, x.Decision
    same = x.Session.eq(x.Session.shift(-1))
    return int(((R_ == 1) & (c_ >= 0) & same & (c_.shift(-1) >= 0) & (c_.shift(-1) != c_) & (R_.shift(-1) == 1)).sum())


for lab, x in [("(a) S38, sessions with a dopamine response", d[d.Session.isin(S38)]),
               ("(b) S46, all recorded sessions", d[d.Session.isin(S46)]),
               ("(b') all 48 sessions with a photometry file", d[d.Session.isin(meta[meta.has_fp].Session)]),
               ("(c) all 69 sessions", d)]:
    print(f"{lab}: {rsr(x)} reward-switch-reward trials out of {len(x)}")

**Result: reproduced, resting on an assumption (the session set).** In the 38 sessions with a
dopamine response there are exactly 94 trials
that start a reward, switch, reward sequence, out of 33,415 trials, within tolerance of 30,707
(reading a, as the main session set is taken). All 69 sessions give 157.

## fig4bc-reward-omission, fig4bc-switch-effect, fig4bc-no-laterality

> Rewards are followed by an increase of NAc dLight signal (estimate 1.6345) and omissions by a
> decrease (estimate -1.5840).

> Outcome dopamine is shifted upward on switch trials (coefficient 0.5030); reward responses are
> larger when mice have just switched ports, and omission dips are larger when they stayed.

> Choice laterality relative to the recorded hemisphere has no significant effect on outcome
> dopamine (p = 0.523).

OLS `DA ~ 0 + reward + omission + switch + contra` (claim). Readings: response (a) DA-PT, (b) mean
de-based Z(DA) over 0 to 1 s; trials (i) cleaned (SO_lat <= 1 s), (ii) all trials with a choice and
a previous choice. Main session set S38; S46 and the Lasso preprocessing are variants.

In [ ]:
def four_factor(N, sessions_label):
    rows = []
    for tl, T in [("(i) cleaned", q.cleaned(N, 1.0)), ("(ii) all trials", N[N.valid])]:
        for yl, y in [("(a) DA_PT", "DA_PT"), ("(b) DA_mean", "DA_mean")]:
            T2 = T.dropna(subset=[y, "switch"])
            m = smf.ols(f"{y} ~ 0 + reward + omission + switch + contra", T2).fit()
            ci = m.conf_int()
            r = {"sessions": sessions_label, "trials": tl, "response": yl, "n": len(T2)}
            for k in ["reward", "omission", "switch", "contra"]:
                r[k] = m.params[k]; r[k + "_lo"] = ci.loc[k, 0]; r[k + "_hi"] = ci.loc[k, 1]; r[k + "_p"] = m.pvalues[k]
            rows.append(r)
    return pd.DataFrame(rows)


ff = pd.concat([four_factor(N38, "S38"), four_factor(N46, "S46")])
show = ["sessions", "trials", "response", "n", "reward", "reward_lo", "reward_hi", "omission", "omission_lo",
        "omission_hi", "reward_p", "omission_p"]
print(ff[show].round(4).to_string(index=False))
print("paper: reward 1.6345 [1.613, 1.656], omission -1.5840 [-1.604, -1.564], p <= 1e-5")

In [ ]:
# Lasso preprocessing variant (Martianova et al. 2019), S38
def zlasso(s):
    t, z, _, _ = q.preprocess(RAW[s], regression="lasso")
    return s, t, z


with ThreadPoolExecutor(4) as ex:
    ZL = {s: (t, z) for s, t, z in ex.map(zlasso, sorted(S38))}
rows = []
for s in sorted(S38):
    x = N38[N38.Session == s].copy()
    g, y = q.outcome_aligned(ZL[s][0], ZL[s][1], x.outcome.values, 1.0)
    x["DA_PT"], x["DA_mean"] = q.da_measures(g, y, x.Reward.values)
    rows.append(x)
N38L = pd.concat(rows)
ffl = four_factor(N38L, "S38 Lasso")
print(ffl[show + ["switch", "switch_lo", "switch_hi", "contra", "contra_p"]].round(4).to_string(index=False))

**Result: partially reproduced, resting on an assumption (the session set).** Rewards raise and omissions lower outcome dopamine with p far
below 1e-5 under every reading, but no reading puts both estimates inside the paper's intervals. On
cleaned trials with DA-PT (S38), the reward estimate is 1.717 [1.692, 1.742] and the omission
estimate -1.559 [-1.584, -1.535], against 1.6345 [1.613, 1.656] and -1.584 [-1.604, -1.564]; on all
trials the omission estimate is inside (-1.587) and the reward estimate further out (1.927). The
mean over 0 to 1 s gives much smaller values (0.47 and -0.70). With all 46 sessions (cleaned, DA-PT)
the reward estimate is 1.683 and the omission -1.513; with the Lasso preprocessing 1.731 and -1.555.
All p values are printed as 0.

In [ ]:
print(ff[["sessions", "trials", "response", "n", "switch", "switch_lo", "switch_hi", "switch_p"]].round(4).to_string(index=False))
print("paper: switch 0.5030 [0.454, 0.552]")
for sl, N in [("S38", N38), ("S46", N46)]:
    for tl, T in [("(i) cleaned", q.cleaned(N, 1.0)), ("(ii) all trials", N[N.valid])]:
        mt = T.dropna(subset=["switch"]).groupby(["reward", "switch"]).DA_PT.mean()
        print(sl, tl, "mean DA_PT: rewarded stay %.3f, rewarded switch %.3f, unrewarded stay %.3f, unrewarded switch %.3f"
              % (mt[(1, 0)], mt[(1, 1)], mt[(0, 0)], mt[(0, 1)]))

**Result: reproduced, resting on an assumption (the session set).** On cleaned trials with DA-PT
(S38) the switch coefficient is 0.517 [0.483, 0.551], p printed as 0, inside [0.454, 0.552].
Rewarded switch trials have a larger mean DA-PT than rewarded stay trials (2.345 against 1.669), and
unrewarded stay trials a deeper dip than unrewarded switch trials (-1.564 against -1.132); the same
holds under every reading and session set. On all trials the coefficient is 0.637, outside; with all
46 sessions and cleaned trials it is 0.467, inside.

In [ ]:
print(ff[["sessions", "trials", "response", "n", "contra", "contra_lo", "contra_hi", "contra_p"]].round(4).to_string(index=False))
print("paper: p = 0.523")

**Result: reproduced, resting on an assumption.** The laterality coefficient is small under every
reading (-0.003 to -0.050 z, against a reward coefficient near 1.7), but its p depends on the trials
used. The paper fits this OLS for its Fig 4B and 4C, for which the Methods describe only dropping
rows with missing entries; the cleaning is described for the RPE regression. On all trials with
DA-PT (reading a-ii, the one adopted) p = 0.234 in S38 and 0.816 in S46; on cleaned trials
p = 0.0003 in S38 and 0.010 in S46, and the 0 to 1 s mean on all S38 trials gives p = 0.144.

## fig4de-port-duration

> The duration of the outcome dopamine response follows the time the mouse stays at the reward
> port, with the trough after omissions reached just after the mouse leaves.

In [ ]:
def port_measures(N):
    T = q.cleaned(N, 1.0)
    Y = np.vstack(T.trace.values)
    rw = T.Reward.values == 1
    # (1) rewarded: first time after the peak (0 to 1 s) that de-based Z falls below half the peak
    w1 = (GRID > 0) & (GRID <= 1.0 + 1e-9)
    half = np.full(len(T), np.nan)
    for i in np.flatnonzero(rw):
        y = Y[i]
        j = np.flatnonzero(w1)[np.nanargmax(y[w1])]
        pk = y[j]
        below = np.flatnonzero(y[j:] < pk / 2)
        if len(below):
            half[i] = GRID[j + below[0]]
    # (2) unrewarded: time of the minimum in 0 to 6 s
    trough = np.where(~rw, GRID[np.nanargmin(np.where(np.isnan(Y), np.inf, Y), axis=1)], np.nan)
    return T.port_dur.values, rw, half, trough


for sl, N in [("S38", N38), ("S46", N46)]:
    pdur, rw, half, trough = port_measures(N)
    k1 = rw & ~np.isnan(half)
    r1 = stats.spearmanr(pdur[k1], half[k1])
    k2 = ~rw & ~np.isnan(trough)
    r2 = stats.spearmanr(pdur[k2], trough[k2])
    lag = np.median(trough[k2] - pdur[k2])
    print(f"{sl}: (1) rewarded n={k1.sum()} (no half-decay within 6 s: {(rw & np.isnan(half)).sum()}), "
          f"Spearman rho = {r1.statistic:.3f}, p = {r1.pvalue:.2g}; median half-decay {np.median(half[k1]):.2f} s")
    print(f"     (2) unrewarded n={k2.sum()}, Spearman rho = {r2.statistic:.3f}, p = {r2.pvalue:.2g}; "
          f"median trough - port_dur = {lag:.3f} s; median port_dur {np.median(pdur[k2]):.2f} s")

**Result: partially reproduced.** On rewarded trials the time at which de-based Z(DA) falls to
half its peak rises with port_dur (Spearman rho 0.467, p printed as 0, S38). On unrewarded trials
the time of the trough also rises with port_dur, weakly (rho 0.062, p = 8.6e-12), but the trough
comes a median 1.70 s after the mouse leaves the port, not within 1 s. S46 gives the same pattern
(0.426; 0.038; 1.81 s).

## Model RPEs and the dopamine regressions

Each model, fitted per mouse to behaviour, is run on the mouse's own choices and rewards; the RPE of
trial t is r_t - Q_t(c_t) (code). BIfp is also run with the Methods' RPE (`BIfp_methods`) and RLFQ3p
with the Methods' forgetting rule (`RLFQ3p_methods`).

In [ ]:
RPE_MODELS = ["RL4p", "RLCF", "RLFQ3p", "RLFQ3p_methods", "RL_meta", "PearceHall", "BIfp", "BIfp_methods",
              "BRLfwr", "BRLfw", "BRLwrp"]
for s, g in d.groupby("Subject"):
    gg = g.reset_index(drop=True)
    for m in RPE_MODELS:
        if m == "BIfp_methods":
            r = q.bifp_methods_rpe(FPAR[("BIfp", s)], gg)
        else:
            _, r = q.model_latents(m, FPAR[(m, s)], gg)
        d.loc[g.index, "rpe_" + m] = r
d["R_chosen"], d["R_unchosen"], d["hist_ok"] = q.history_counts(d, 4)
add = ["Session", "Trial", "R_chosen", "R_unchosen", "hist_ok"] + ["rpe_" + m for m in RPE_MODELS]
N38 = N38.drop(columns=[c for c in add[2:] if c in N38]).merge(d[add], on=["Session", "Trial"])
N46 = N46.drop(columns=[c for c in add[2:] if c in N46]).merge(d[add], on=["Session", "Trial"])
C38 = q.cleaned(N38, 1.0)
C38 = C38[(C38.MVMT > 0) & (C38.center_dur > 0)].reset_index(drop=True)
print("cleaned S38 trials:", len(C38), "| dropped for log of a non-positive duration:",
      len(q.cleaned(N38, 1.0)) - len(C38))
print(C38[["rpe_" + m for m in RPE_MODELS]].describe().T[["mean", "min", "max"]].round(3))

## fig4f-bayesian-rpe-fit, figs3-other-metrics, figs8-brlwrp-no-difference

> RPEs of BIfp and BRLfwr explain outcome dopamine better than RL4p RPEs and better than RLCF,
> RLFQ3p, RL_meta and PearceHall, by cross-validated log-likelihood.

Per animal, 70% of trials fit the Methods' regression and 30% test it; the test log-likelihood uses
the training residual variance. Readings (claim): (a) random trial splits, 100 repeats, Delta summed
over animals per repeat, bootstrap over repeats; (b) Delta per animal, bootstrap over animal and
repeat pairs; (c) splits by session (30% of each animal's sessions held out), 100 repeats, as (b).

In [ ]:
y38 = C38.DA_PT.values
X38 = {m: q.rpe_design(C38, C38["rpe_" + m].values) for m in RPE_MODELS}
subj = C38.Subject.values
sess = C38.Session.values


def cv_runs(split, n_rep, seed):
    rng = np.random.default_rng(seed)
    out = []
    for rep in range(n_rep):
        for s in np.unique(subj):
            idx = np.flatnonzero(subj == s)
            if split == "trial":
                perm = rng.permutation(idx)
                k = int(round(0.7 * len(idx)))
                tr, te = perm[:k], perm[k:]
            else:
                ss = rng.permutation(np.unique(sess[idx]))
                k = int(round(0.7 * len(ss)))
                tr = idx[np.isin(sess[idx], ss[:k])]
                te = idx[np.isin(sess[idx], ss[k:])]
            for m in RPE_MODELS:
                llk, b = q.llk_cv(X38[m], y38, tr, te)
                e = y38[te] - X38[m][te] @ b
                r2 = 1 - np.sum(e ** 2) / np.sum((y38[te] - y38[te].mean()) ** 2)
                out.append((rep, s, m, llk, r2))
    return pd.DataFrame(out, columns=["rep", "s", "m", "llk", "r2"])


cvt = cv_runs("trial", 100, 105)
cvs = cv_runs("session", 100, 106)


def boot(vals, rng, n=2000):
    vals = np.asarray(vals)
    return np.percentile([rng.choice(vals, len(vals)).mean() for _ in range(n)], [2.5, 97.5])


rng = np.random.default_rng(107)
tab = {}
for lab, cv, agg in [("(a) trial splits, sum over animals", cvt, "sum"), ("(b) trial splits, per animal", cvt, "animal"),
                     ("(c) session splits, per animal", cvs, "animal")]:
    P = cv.pivot_table(index=["rep", "s"], columns="m", values="llk")
    D = P.sub(P.RL4p, axis=0)
    if agg == "sum":
        D = D.groupby(level=0).sum()
    rows = {}
    for m in RPE_MODELS:
        lo, hi = boot(D[m].values, rng)
        rows[m] = (D[m].mean(), lo, hi)
    tab[lab] = pd.DataFrame(rows, index=["delta_llk", "ci_lo", "ci_hi"]).T
    print(lab)
    print(tab[lab].round(2).T)
    print()
paper = {"BRLfwr": (60.84, 102.79), "BIfp": (65.71, 217.5), "RLCF": (-47.03, 4.64), "RLFQ3p": (4.98, 33.31),
         "RL_meta": (7.96, 33.9), "PearceHall": (6.04, 32.83)}
for lab, t_ in tab.items():
    inside = {m: bool(paper[m][0] <= t_.loc[m, "delta_llk"] <= paper[m][1]) for m in paper}
    order = all(t_.loc[b, "delta_llk"] > t_.loc[o, "delta_llk"] for b in ["BRLfwr", "BIfp"]
                for o in ["RLCF", "RLFQ3p", "RL_meta", "PearceHall"])
    pos = all(t_.loc[b, "ci_lo"] > 0 for b in ["BRLfwr", "BIfp"])
    print(f"{lab}: Bayesian CIs above 0 {pos}, Bayesian above the four RL models {order}, inside paper's intervals {inside}")
    print("   variants: BIfp_methods %.2f, RLFQ3p_methods %.2f" % (t_.loc["BIfp_methods", "delta_llk"], t_.loc["RLFQ3p_methods", "delta_llk"]))

**Result: partially reproduced, resting on an assumption (the session set).** Under every reading
the Delta llk_CV of BRLfwr and BIfp has a CI above 0 and exceeds that of RLCF, RLFQ3p, RL_meta and
PearceHall. Per animal with trial splits (reading b): BRLfwr 75.85, BIfp 115.30, RLCF -21.70,
RLFQ3p 21.91 and RL_meta 30.61 lie inside the paper's intervals, but PearceHall gives -14.17, outside
[6.04, 32.83]; session splits give the same pattern, and summing over animals (reading a) gives
values about five times larger. No reading puts all six inside. With the Methods' BIfp RPE, BIfp
gives 110.70; with the Methods' forgetting rule, RLFQ3p gives 40.79.

In [ ]:
rows = []
for s in np.unique(subj):
    idx = np.flatnonzero(subj == s)
    for m in RPE_MODELS:
        r = sm.OLS(y38[idx], X38[m][idx]).fit()
        rows.append((s, m, r.aic, r.bic))
ab = pd.DataFrame(rows, columns=["s", "m", "aic", "bic"])
met = {}
for k in ["aic", "bic"]:
    P = ab.pivot(index="s", columns="m", values=k)
    met[f"relative {k.upper()} (lower better), sum over animals"] = -(P.sub(P.RL4p, axis=0)).sum()
Pr = cvt.pivot_table(index=["rep", "s"], columns="m", values="r2")
met["relative CV R2 (higher better), mean"] = Pr.sub(Pr.RL4p, axis=0).mean()
Pl = cvt.pivot_table(index=["rep", "s"], columns="m", values="llk")
met["relative CV llk (higher better), mean per animal"] = Pl.sub(Pl.RL4p, axis=0).mean()
M = pd.DataFrame(met)
print("all metrics oriented so that higher is better (AIC and BIC sign-flipped):")
print(M.round(4))
others = ["RL4p", "RLCF", "RLFQ3p", "RL_meta", "PearceHall"]
for col in M:
    ok = all(M.loc[b, col] > M.loc[o, col] for b in ["BIfp", "BRLfwr"] for o in others)
    print(f"{col}: BIfp and BRLfwr above RL4p, RLCF, RLFQ3p, RL_meta, PearceHall: {ok}")

**Result: reproduced, resting on an assumption (the session set).** For relative AIC, relative
BIC (both from the fit to all cleaned trials of each animal, summed), cross-validated R2 and
cross-validated log-likelihood, BIfp and BRLfwr are better than RL4p, RLCF, RLFQ3p, RL_meta and
PearceHall. AIC and BIC give the same ordering, as every model's regression has the same number of
terms.

In [ ]:
for var in ["BRLwrp", "BRLfw"]:
    x = aic[var] - aic["BRLfwr"]
    t, p = stats.ttest_1samp(x, 0)
    print(f"behaviour: AIC({var}) - AIC(BRLfwr) per mouse {x.round(2).to_dict()}, mean {x.mean():.2f}, t = {t:.3f}, p = {p:.4f}")
    for lab, cv, agg in [("(b) trial splits, per animal", cvt, "animal"), ("(a) trial splits, sum over animals", cvt, "sum"),
                         ("(c) session splits, per animal", cvs, "animal")]:
        P = cv.pivot_table(index=["rep", "s"], columns="m", values="llk")
        D = P[var] - P["BRLfwr"]
        if agg == "sum":
            D = D.groupby(level=0).sum()
        lo, hi = boot(D.values, rng)
        print(f"   dopamine {lab}: llk({var}) - llk(BRLfwr) = {D.mean():.2f}, 95% CI [{lo:.2f}, {hi:.2f}]")

**Result: not reproduced.** BRLwrp differs from BRLfwr on both counts. In behaviour its AIC is
lower by 80.31 on average (t = -2.899, p = 0.044); in dopamine its Delta llk_CV is higher by 60.95
per animal, 95% CI [56.66, 65.37], which excludes 0 (all three readings exclude 0). The variant
BRLfw against BRLfwr does not differ in behaviour (p = 0.187) but does in dopamine (29.04, CI
[25.51, 32.46]). The dopamine comparison rests on the session set assumption.

## fig4g-history-decrease, fig4h-nonnegative-rpe-aaa, fig4-consistent-after-reward

> On consecutive stay trials, dopamine responses to reward decrease as recent rewards accumulate
> (from aaa to AAA) but do not flatten to no response.

Fig 4G trials: cleaned rewarded trials t whose choices at t-3, t-2, t-1 and t are at the same port
in the same session; the history is the outcomes at t-3, t-2, t-1.

In [ ]:
c_ = d.Decision.values
s_ = d.Session.values
r_ = np.nan_to_num(d.Reward.values.astype(float))
lab4 = np.array([None] * len(d), dtype=object)
for i in range(3, len(d)):
    if s_[i - 3] == s_[i] and c_[i] >= 0 and c_[i - 1] == c_[i] and c_[i - 2] == c_[i] and c_[i - 3] == c_[i]:
        lab4[i] = "".join("A" if r_[j] == 1 else "a" for j in (i - 3, i - 2, i - 1))
d["hist4g"] = lab4
C38 = C38.drop(columns=[c for c in ["hist4g"] if c in C38]).merge(d[["Session", "Trial", "hist4g"]], on=["Session", "Trial"])
G = C38[(C38.Reward == 1) & C38.hist4g.notna()].copy()
G["n_rew"] = G.hist4g.str.count("A")
ORDER = ["aaa", "aaA", "aAa", "Aaa", "aAA", "AaA", "AAa", "AAA"]
print("Fig 4G trials:", len(G), "| per history:", G.hist4g.value_counts().reindex(ORDER).to_dict())
print(G.groupby("hist4g").DA_PT.agg(["mean", "sem", "size"]).reindex(ORDER).round(3))
r = stats.spearmanr(G.n_rew, G.DA_PT)
print(f"(1) Spearman rho(number of rewards, DA_PT) = {r.statistic:.4f}, p = {r.pvalue:.2g}")
rng = np.random.default_rng(108)
aaa = G[G.hist4g == "AAA"].DA_mean
print(f"(2) AAA trials n = {len(aaa)}: mean de-based Z(DA) over 0 to 1 s = {aaa.mean():.4f}, 95% CI {np.round(q.bootstrap_ci(aaa.values, rng), 4)}")
print("per mouse rho:", {s: round(stats.spearmanr(g.n_rew, g.DA_PT).statistic, 3) for s, g in G.groupby("Subject")})

**Result: reproduced, resting on an assumption (the session set).** On 6,345 Fig 4G trials the
reward response falls as rewards accumulate (Spearman rho between the number of rewards and DA-PT
-0.163, p = 6.6e-39; negative in each mouse), and on AAA trials the mean de-based Z(DA) over 0 to
1 s is 0.327, 95% CI [0.293, 0.359], above 0.

In [ ]:
MR = ["BRLfwr", "BIfp", "RL4p", "RLCF", "RLFQ3p", "RL_meta", "PearceHall"]
mr = G.groupby("hist4g")[["rpe_" + m for m in MR]].mean().reindex(ORDER)
mr.columns = MR
print("mean model RPE per history (Fig 4G trials):")
print(mr.round(4))
a = mr.loc["AAA"]
for b in ["BRLfwr", "BIfp"]:
    print(f"{b}: AAA RPE {a[b]:.4f} > 0: {a[b] > 0}; above every RL model: {all(a[b] > a[o] for o in MR[2:])}")

**Result: reproduced, resting on an assumption (the session set).** On AAA trials the mean RPE is
0.278 for BRLfwr and 0.591 for BIfp, both above 0 and above RL4p, RLCF, RLFQ3p, RL_meta and
PearceHall (0.002 to 0.018).

In [ ]:
A4 = G[G.hist4g.isin(["aaA", "AaA", "aAA", "AAA"])]
f = stats.f_oneway(*[g.DA_PT.values for _, g in A4.groupby("hist4g")])
print(f"(1) one-way ANOVA of DA_PT over aaA, AaA, aAA, AAA: F = {f.statistic:.3f}, p = {f.pvalue:.3g}; means",
      A4.groupby("hist4g").DA_PT.mean().round(3).to_dict())
rng4 = (mr.loc[["aaA", "AaA", "aAA", "AAA"]].max() - mr.loc[["aaA", "AaA", "aAA", "AAA"]].min()) / (mr.max() - mr.min())
print("(2) range over the four histories / range over all eight, per model:")
print(rng4.round(4).sort_values().to_string())
print("smallest:", rng4.idxmin())

**Result: partially reproduced, resting on an assumption (the session set).** DA-PT does not
differ across aaA, AaA, aAA and AAA (one-way ANOVA F = 1.675, p = 0.17; means 1.509 to 1.601).
But BRLfwr is not the model whose RPE is most constant over these histories: relative to its range
over all eight histories, BIfp varies least (0.0007), then RL4p (0.144), then BRLfwr (0.192).

## fig4h-rpe-only-llk

> OLS models predicting dopamine from each model's RPE alone fit best for BRLfwr (cross-validated
> log-likelihood -985.89, against -991.80 to -995.02 for the RL models).

In [ ]:
H_RPE = ["BRLfwr", "RL4p", "RLCF", "PearceHall", "RLFQ3p", "RL_meta", "BIfp"]
rng = np.random.default_rng(109)
rows = []
for tl, T in [("(i) Fig 4G trials", G.reset_index(drop=True)), ("(ii) all cleaned", C38)]:
    yy = T.DA_PT.values
    sj = T.Subject.values
    for fl in ["(a) rpe:pos + rpe:neg", "(b) one slope"]:
        Xs = {}
        for m in H_RPE:
            rp = T["rpe_" + m].values
            if fl.startswith("(a)"):
                pos = (rp >= 0).astype(float)
                Xs[m] = np.column_stack([np.ones(len(T)), rp * pos, rp * (1 - pos)])
            else:
                Xs[m] = np.column_stack([np.ones(len(T)), rp])
        res = {m: [] for m in H_RPE}
        for rep in range(100):
            for s in np.unique(sj):
                idx = np.flatnonzero(sj == s)
                perm = rng.permutation(idx)
                k = int(round(0.7 * len(idx)))
                for m in H_RPE:
                    res[m].append((rep, s, q.llk_cv(Xs[m], yy, perm[:k], perm[k:])[0]))
        for m in H_RPE:
            x = pd.DataFrame(res[m], columns=["rep", "s", "llk"])
            rows.append((tl, fl, m, x.llk.mean(), x.groupby("rep").llk.sum().mean()))
h4 = pd.DataFrame(rows, columns=["trials", "terms", "model", "llk per animal", "llk summed over animals"])
print(h4.pivot_table(index="model", columns=["trials", "terms"], values="llk per animal").reindex(H_RPE).round(2))
print(h4.pivot_table(index="model", columns=["trials", "terms"], values="llk summed over animals").reindex(H_RPE).round(2))
for (tl, fl), g in h4.groupby(["trials", "terms"]):
    g6 = g[g.model != "BIfp"].set_index("model")["llk per animal"]
    print(tl, fl, "highest of the six:", g6.idxmax(), "| order:", list(g6.sort_values(ascending=False).index))
print("paper: BRLfwr -985.89, RL4p -991.80, RLCF -992.87, PearceHall -993.81, RLFQ3p -994.74, RL_meta -995.02")

**Result: not reproduced.** The order depends on the trials. On the Fig 4G trials, RL4p has the
highest cross-validated log-likelihood (-572.64 per animal with the split RPE terms) and BRLfwr the
lowest (-575.40), under both RPE codings; on all cleaned trials BRLfwr is highest (-1963.88) and
BIfp, outside the six, higher still (-1853.70). The criterion asks the order under every reading. No
reading gives values near the paper's (about -986 to -995): per animal they are near -573 or
-1964 to -2350, and summed over animals near -2870 or -9800 to -11750. The result rests on the
session set assumption.

## fig4i-lmer-switch, fig4j-lmer-stay

> On switch trials, past rewards at the chosen port reduce outcome dopamine (slope -0.128) and past
> rewards at the other port increase it (slope 0.126).

> On stay trials, past rewards at the chosen port reduce outcome dopamine (slope -0.09), as all
> models predict.

The Methods' formula, fitted with statsmodels' MixedLM (lme4 is not in this environment; both fit
the same model class by REML). Readings of the random part: (a) correlated random intercept and
slopes for all terms by Subject; (b) uncorrelated random slopes plus a random intercept. Trials:
cleaned S38 trials whose 4 previous trials are in the session.

In [ ]:
L = C38[C38.hist_ok & C38.Switch.notna()].copy()
TERMS = ["RwSw", "RcSw", "Switch", "RuSw", "RwSt", "RcSt", "RuSt"]


def add_terms(x):
    x = x.copy()
    x["Stay"] = 1 - x["Switch"]
    x["Rw"] = (x["Reward"] == 1).astype(float)
    x["RwSw"], x["RcSw"], x["RuSw"] = x.Rw * x.Switch, x.R_chosen * x.Switch, x.R_unchosen * x.Switch
    x["RwSt"], x["RcSt"], x["RuSt"] = x.Rw * x.Stay, x.R_chosen * x.Stay, x.R_unchosen * x.Stay
    return x


L = add_terms(L)
print("trials:", len(L), "| switch trials:", int(L.Switch.sum()), "| per mouse:", L.groupby("Subject").size().to_dict())
print("R_chosen range", L.R_chosen.min(), L.R_chosen.max(), "| R_unchosen range", L.R_unchosen.min(), L.R_unchosen.max())
FORM = " + ".join(TERMS)
lm_rows = []
for yl in ["DA_PT", "DA_mean"]:
    for rl in ["(a) correlated", "(b) uncorrelated"]:
        try:
            if rl.startswith("(a)"):
                mm = smf.mixedlm(f"{yl} ~ {FORM}", L, groups="Subject", re_formula="~" + FORM).fit()
            else:
                mm = smf.mixedlm(f"{yl} ~ {FORM}", L, groups="Subject", re_formula="1",
                                 vc_formula={k: "0 + " + k for k in TERMS}).fit()
            ci = mm.conf_int()
            for k in ["RcSw", "RuSw", "RcSt"]:
                lm_rows.append((yl, rl, k, mm.params[k], ci.loc[k, 0], ci.loc[k, 1], mm.pvalues[k], mm.converged))
        except Exception as e:
            lm_rows.append((yl, rl, "failed: " + type(e).__name__, np.nan, np.nan, np.nan, np.nan, False))
lmr = pd.DataFrame(lm_rows, columns=["DA", "random part", "term", "estimate", "ci_lo", "ci_hi", "p", "converged"])
print(lmr.round(4).to_string(index=False))
print("paper: R_chosen:Switch -0.128 [-0.195, -0.062]; R_unchosen:Switch 0.126 [0.035, 0.217]; R_chosen:Stay -0.09 [-0.11, -0.07]")
ols = smf.ols(f"DA_PT ~ {FORM}", L).fit()
print("\nOLS without subject effects (DA_PT):", ols.params[["RcSw", "RuSw", "RcSt"]].round(4).to_dict())

**Result (fig4i-lmer-switch): reproduced, resting on an assumption (the session set).** With
correlated random effects and DA-PT (reading a-i) R_chosen:Switch is -0.165 [-0.216, -0.115] and
R_unchosen:Switch 0.100 [0.036, 0.164], both excluding 0 and inside the paper's intervals
[-0.195, -0.062] and [0.035, 0.217]; every other reading gives the same.

## fig4i-model-runchosen

> On switch trials BIfp, BRLfwr and RLCF predict a positive effect of past rewards at the other port
> on the RPE, while the other RL models predict little or none.

Model RPEs are regressed on the fixed part of the formula by OLS. Readings: (a) the code's
generate with each mouse's parameters, one simulated session per real session of the same length,
on blocks of 7 to 23 or 8 to 15 rewards; (b) RPEs of the models run on the mice's own trials (all
69 sessions, trials whose 4 previous trials are in the session).

In [ ]:
RU_MODELS = ["BIfp", "BRLfwr", "RLCF", "RL4p", "RLFQ3p", "RL_meta", "PearceHall"]


def ols_terms(x, ycol):
    x = add_terms(x[x.hist_ok & x.Switch.notna() & x[ycol].notna()])
    return smf.ols(f"{ycol} ~ {FORM}", x).fit()


res = {}
dd = d[d.valid].copy()
dd["Switch"] = d.loc[dd.index, "Switch"]
for m in RU_MODELS:
    r = ols_terms(dd, "rpe_" + m)
    res[("(b) mice trials", m)] = (r.params["RuSw"], r.pvalues["RuSw"], r.params["RcSw"], r.params["RcSt"])
lens = d.groupby(["Subject", "Session"]).size()
for blocks in [(7, 23), (8, 15)]:
    for m in RU_MODELS:
        sims = []
        k = 0
        for (s, sess_), n_tr in lens.items():
            sim, _ = q.generate_frame(m, FPAR[(m, s)], int(n_tr), 1, blocks, 500 + k, ident=f"{s}_{k}")
            k += 1
            _, rp = q.model_latents(m, FPAR[(m, s)], sim)
            sim["rpe"] = rp
            sims.append(sim)
        sim = pd.concat(sims, ignore_index=True)
        same = sim.Session.eq(sim.Session.shift(1))
        sim["Switch"] = np.where(same, (sim.Decision != sim.Decision.shift(1)).astype(float), np.nan)
        sim["R_chosen"], sim["R_unchosen"], sim["hist_ok"] = q.history_counts(sim, 4)
        r = ols_terms(sim, "rpe")
        res[(f"(a) generate {blocks[0]}-{blocks[1]}", m)] = (r.params["RuSw"], r.pvalues["RuSw"], r.params["RcSw"], r.params["RcSt"])
ru = pd.DataFrame(res, index=["R_unchosen:Switch", "p", "R_chosen:Switch", "R_chosen:Stay"]).T
print(ru.round(4))
for rd, g in ru.groupby(level=0):
    g = g.droplevel(0)
    pos3 = all((g.loc[m, "R_unchosen:Switch"] > 0) and (g.loc[m, "p"] < 0.05) for m in ["BIfp", "BRLfwr", "RLCF"])
    small = 0.5 * g.loc[["BIfp", "BRLfwr", "RLCF"], "R_unchosen:Switch"].min()
    little = {m: bool(not (g.loc[m, "R_unchosen:Switch"] > 0 and g.loc[m, "p"] < 0.05) or g.loc[m, "R_unchosen:Switch"] < small)
              for m in ["RL4p", "RLFQ3p", "RL_meta", "PearceHall"]}
    print(f"{rd}: BIfp, BRLfwr, RLCF positive with p < 0.05: {pos3}; little or none for {little}")

**Result: reproduced.** Under every reading, R_unchosen:Switch is positive with p < 0.05 for BIfp
(0.051 to 0.056), BRLfwr (0.066 to 0.069) and RLCF (0.140 to 0.149), and for RL4p, RLFQ3p, RL_meta
and PearceHall it is either not significantly positive or below half the smallest of those three
(at most 0.0086, against a half of 0.026).

In [ ]:
print("fig4j: every model's R_chosen:Stay coefficient negative under each reading:")
for rd, g in ru.groupby(level=0):
    print(rd, (g["R_chosen:Stay"] < 0).all(), g["R_chosen:Stay"].droplevel(0).round(4).to_dict())

**Result (fig4j-lmer-stay): reproduced, resting on an assumption (the session set).** R_chosen:Stay
is -0.099 [-0.141, -0.058] (reading a-i), inside [-0.11, -0.07], and every model's R_chosen:Stay
coefficient is negative (printed above).

## figs5-per-animal

> Dopamine results are qualitatively consistent across animals; per-animal regressions on switch
> trials resemble the Bayesian model predictions in all but one animal, and switch-stay differences
> in outcome responses are consistent across animals.

In [ ]:
rows = []
for s, g in L.groupby("Subject"):
    r = smf.ols(f"DA_PT ~ {FORM}", g).fit()
    mt = g.groupby(["Rw", "Switch"]).DA_PT.mean()
    rows.append((s, len(g), r.params["RcSw"], r.pvalues["RcSw"], r.params["RuSw"], r.pvalues["RuSw"],
                 mt[(1, 1)], mt[(1, 0)], mt[(0, 0)], mt[(0, 1)]))
p5 = pd.DataFrame(rows, columns=["Subject", "n", "R_chosen:Switch", "p_c", "R_unchosen:Switch", "p_u",
                                 "rew_switch", "rew_stay", "unrew_stay", "unrew_switch"])
p5["bayes_pattern"] = (p5["R_chosen:Switch"] < 0) & (p5["R_unchosen:Switch"] > 0)
p5["switch_stay_pattern"] = (p5.rew_switch > p5.rew_stay) & (p5.unrew_stay < p5.unrew_switch)
print(p5.round(4).to_string(index=False))
print("animals with the Bayesian sign pattern:", int(p5.bayes_pattern.sum()), "| with the S5A switch-stay pattern:",
      int(p5.switch_stay_pattern.sum()))

**Result: reproduced, resting on an assumption (the session set).** Four of the five animals show
a negative R_chosen:Switch and a positive R_unchosen:Switch; BSD016 has R_unchosen:Switch -0.007
(p = 0.85). In all five, rewarded switch trials exceed rewarded stay trials and unrewarded stay
trials are below unrewarded switch trials.

## figs4-lmer-feature-sets

> In cross-validated R2 of dopamine regressions, the 3R+Sw feature set captures a similar level of
> variance to L0:R, the standard set of trial-back choice and reward interactions; 4 lags were used
> at the expense of about 1% of variance explained.

L0:R at L lags: the reward at t and, for k = 1..L, the choice at t-k coded +1 when at the port chosen
at t and -1 otherwise, the reward at t-k, and their product (assumption: the legend does not code
the choice; coding it relative to the current choice keeps the set free of laterality, which the
four-factor OLS found negligible). 3R+Sw at L lags: the fixed part of the fig4i formula with
R_chosen and R_unchosen counted over L lags. Cleaned S38 trials with L previous trials in the session
(L = 8, so that every set uses the same trials). Readings: (a) OLS, (b) the uncorrelated MixedLM, predicting the test trials from its fixed effects;
70-30 random splits within each animal, test R2 over the pooled test trials; 20 repeats for OLS and
3 for MixedLM (time).

In [ ]:
LMAX = 8
X = C38.copy()
c_all, r_all, s_all = d.Decision.values, np.nan_to_num(d.Reward.values.astype(float)), d.Session.values
pos = d.reset_index(drop=True).set_index(["Session", "Trial"]).index.get_indexer(list(zip(X.Session, X.Trial)))
ok = np.ones(len(X), bool)
for k in range(1, LMAX + 1):
    j = pos - k
    same = (j >= 0) & (s_all[np.maximum(j, 0)] == X.Session.values)
    ok &= same
    X[f"C{k}"] = np.where(c_all[np.maximum(j, 0)] == X.Decision.values, 1.0, -1.0)
    X[f"R{k}"] = r_all[np.maximum(j, 0)]
    X[f"CR{k}"] = X[f"C{k}"] * X[f"R{k}"]
X = X[ok & X.Switch.notna()].reset_index(drop=True)
X["Rw"] = (X.Reward == 1).astype(float)
X["Stay"] = 1 - X.Switch
for Lg in range(1, LMAX + 1):
    rc = sum(((X[f"C{k}"] == 1) & (X[f"R{k}"] == 1)).astype(float) for k in range(1, Lg + 1))
    ru_ = sum(((X[f"C{k}"] == -1) & (X[f"R{k}"] == 1)).astype(float) for k in range(1, Lg + 1))
    X[f"RcSw{Lg}"], X[f"RuSw{Lg}"] = rc * X.Switch, ru_ * X.Switch
    X[f"RcSt{Lg}"], X[f"RuSt{Lg}"] = rc * X.Stay, ru_ * X.Stay
X["RwSw"], X["RwSt"] = X.Rw * X.Switch, X.Rw * X.Stay
print("trials with 8 previous trials in the session:", len(X))


def feats(kind, Lg):
    if kind == "L0:R":
        return ["Rw"] + [f"{p}{k}" for k in range(1, Lg + 1) for p in ("C", "R", "CR")]
    return ["RwSw", f"RcSw{Lg}", "Switch", f"RuSw{Lg}", "RwSt", f"RcSt{Lg}", f"RuSt{Lg}"]


def cv_r2(kind, Lg, reading, n_rep, seed):
    rng = np.random.default_rng(seed)
    out = []
    F = [f for f in feats(kind, Lg) if X[f].std() > 0]  # at 1 lag, R_chosen on switch and R_unchosen on stay are 0
    for rep in range(n_rep):
        tr = np.zeros(len(X), bool)
        for s in X.Subject.unique():
            idx = np.flatnonzero(X.Subject.values == s)
            tr[rng.permutation(idx)[:int(round(0.7 * len(idx)))]] = True
        A, B = X[tr], X[~tr]
        if reading == "OLS":
            r = sm.OLS(A.DA_PT, sm.add_constant(A[F])).fit()
            pred = r.predict(sm.add_constant(B[F]))
        else:
            mm = smf.mixedlm("DA_PT ~ " + " + ".join(F), A, groups="Subject", re_formula="1",
                             vc_formula={f: "0 + " + f for f in F}).fit()
            pred = mm.predict(B)  # fixed effects
        out.append(1 - np.sum((B.DA_PT - pred) ** 2) / np.sum((B.DA_PT - B.DA_PT.mean()) ** 2))
    return np.mean(out)


t0 = time.time()
r2 = pd.DataFrame({(rd, kind): {Lg: cv_r2(kind, Lg, rd, 20 if rd == "OLS" else 3, 110 + Lg)
                                for Lg in range(1, LMAX + 1)}
                   for rd in ["OLS", "MixedLM"] for kind in ["L0:R", "3R+Sw"]})
print("cross-validated R2 by number of lags (time %.0f s):" % (time.time() - t0))
print(r2.round(4))
for rd in ["OLS", "MixedLM"]:
    a, b = r2[(rd, "3R+Sw")], r2[(rd, "L0:R")]
    print(f"{rd}: 3R+Sw(4) - L0:R(4) = {a[4] - b[4]:+.4f}; 3R+Sw best - 3R+Sw(4) = {a.max() - a[4]:+.4f}; "
          f"L0:R best - 3R+Sw(4) = {b.max() - a[4]:+.4f}")

**Result: reproduced, resting on assumptions (the session set and the L0:R choice coding).** At 4
lags the cross-validated R2 of 3R+Sw is within 0.01 of L0:R (OLS -0.0051, MixedLM -0.0050), and
within 0.01 of the best over lags of either set (OLS 0.0044 and 0.0063, MixedLM 0.0000 and 0.0050).
The R2 values (about 0.72) are dominated by the reward term.

## Summary

| Claim | Outcome |
|---|---|
| data-composition | partially reproduced |
| perf-after-pretraining | reproduced |
| fig1f-performance-increase | reproduced |
| trials-to-switch | reproduced |
| trials-to-commit | reproduced |
| figs1-switch-measures-decrease | reproduced |
| fig1e-stay-after-omission | reproduced |
| figs6a-decaying-history-weights | reproduced |
| fig2c-bayesian-beat-rl4p | reproduced |
| fig2c-complex-rl-beat-rl4p | reproduced |
| fig2c-rlcf | reproduced |
| fig2c-no-bayes-vs-complex-rl | reproduced |
| fig2b-model-identifiability | reproduced |
| figs2-parameter-recovery | partially reproduced |
| brlfw-alpha-zero | reproduced |
| figs8-brlwrp-no-difference | not reproduced |
| figs7-no-change-across-sessions | not reproduced |
| fig2f-adaptation-after-switch | reproduced |
| fig3b-sse | partially reproduced |
| fig3a-both-port-contexts | reproduced |
| fig3c-aic-around-switch | reproduced |
| rl4p-learning-rates | reproduced |
| fig4bc-reward-omission | partially reproduced (assumption) |
| fig4bc-switch-effect | reproduced (assumption) |
| fig4bc-no-laterality | reproduced (assumption) |
| fig4de-port-duration | partially reproduced (assumption) |
| trial-cleaning-counts | partially reproduced (assumption) |
| fig4f-bayesian-rpe-fit | partially reproduced (assumption) |
| figs3-other-metrics | reproduced (assumption) |
| fig4g-history-decrease | reproduced (assumption) |
| fig4h-rpe-only-llk | not reproduced (assumption) |
| fig4h-nonnegative-rpe-aaa | reproduced (assumption) |
| fig4-consistent-after-reward | partially reproduced (assumption) |
| fig4i-lmer-switch | reproduced (assumption) |
| fig4j-lmer-stay | reproduced (assumption) |
| fig4i-model-runchosen | reproduced |
| figs5-per-animal | reproduced (assumption) |
| reward-switch-reward-count | reproduced (assumption) |
| figs4-lmer-feature-sets | reproduced (assumptions) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The cognitive models are a numba re-implementation of the paper's code at commit 589555c; on
  the dataset's trials it gives the code's likelihood to 5e-11 and its RPEs to 3e-15. Fits use one
  start, as the code does; the per-mouse AICs match the paper's means closely except PearceHall
  (-678 against -656).
- The paper's 34 sessions with valid dopamine cannot be identified: its AUC rule, as the claims file
  reads it, keeps no session, and the files do not mark valid sessions. The dopamine claims use the
  38 sessions with an outcome response (assumption), with all 46 usable sessions as a variant where
  the claim's own readings are run.
- Z(DA) uses airPLS with the parameters of the cited procedure (Martianova et al. 2019) and a Huber
  robust regression (the loss is not given); the Lasso procedure of the same reference gives
  coefficients within 0.02 of it.
- lme4 is replaced by statsmodels' MixedLM. The S4 Fig choice coding of L0:R is an assumption.

What was found about the dataset:

- 69 sessions, not 70: BSD017 has 13 (days 165 and 166 absent).
- 8 of BSD011's 14 photometry sessions show no outcome response at any clock offset up to 400 s;
  two of them are sampled at 30.5 Hz, the others at 20.4 Hz. Two BSD016 files hold 30 and 40
  samples with timestamps near 5e7 s. Nothing in the files marks which sessions are valid.
- BSD011 was recorded in all 14 sessions; the other mice follow the left, right, no-recording cycle.
- The fiber table gives the same coordinates (AP 1.2, ML 1.2, DV 4.1) for left and right NAc
  sessions; the indicator is described as a calcium indicator and filter wavelengths are -1.
- `stop_time` equals the session's end (`tmax`) on every trial.
- Each photometry series starts with isolated samples up to 140 s before the continuous recording.
- The trial events and photometry share a clock in the 38 sessions with a response (the reward
  contrast appears only at the true event times).

In [ ]:
print("total run time %.1f min" % ((time.time() - T_START) / 60))